### 신경망 모델 구성하기
- 신경망은 데이터에 대한 연산을 수행하는 __layer/module__로 구성되어있음
- `torch.nn` 는 신경망을 구성하는데 필요한 모든 구성 요소를 제공함
- PyTorch의 모든 모듈은 `nn.Module`의 하위 클래스 임

In [60]:
import os
import torch
from torch import nn
from torch.utils.data import Dataset
from torchvision import datasets, transforms

__학습을 위한 장치 설정__
- 가능한 경우 GPU, MPS와 같은 하드웨어 가속기에 모델을 학습하는 것이 좋음
- `torch.cuda` 혹은 `torch.backends.mps`가 사용 가능한지 확인한 후 없으면 CPU를 계속 사용

In [ ]:
device = (
    "cuda" 
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device}")

Using device: mps


__클래스 정의__
- 신경망 모델은 `nn.Module`의 하위 클래스로 정의하고, `__init__`에서 신경망 계층들을 초기화
- `nn.Module`을 상속받은 모든 클래스는 `forward` 메소드에 입력 데이터에 대한 연산들을 구현함

In [14]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [61]:
# Neural Network 의 인스턴스를 생성 -> device로 이동
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


- 모델을 사용하기 위해 입력 데이터를 전달
- 일부 background 연산들과 함께 모델의 `forward`를 실행
    - `model.forward()`를 직접 호출하면 안됨!

In [ ]:
X = torch.rand(1, 28, 28, device=device) # 임의의 입력 데이터 생성
logits = model(X) # 모델에 입력 데이터 전달 
pred_probab = nn.Softmax(dim=1)(logits) # 예측 확률 계산
y_pred = pred_probab.argmax(1) # 가장 높은 확률을 가진 클래스 선택
print(f"Predicted class: {y_pred}") 

Predicted class: tensor([4], device='mps:0')


__Model Layer__
- 모델의 계층을 살펴보기 위해 28 * 28 크기의 이미지 3개로 구성된 미니배치를 호출 후 신경망을 통과할 때 어떤 일이 발생하는지 알아봄

In [45]:
input_image = torch.rand(3, 28, 28)
print(input_image.size())

torch.Size([3, 28, 28])


__nn.Flatten__
- `nn.Flatten` 계층을 초기화 후 28 * 28 size의 2D 이미지를 784 픽셀 값을 갖는 연속된 배열로 반환

In [ ]:
flatten = nn.Flatten() # nn.Flatten 계층 초기화
flat_image = flatten(input_image) # 이미지 평탄화
print(flat_image.size())

torch.Size([3, 784])


__nn.Linear__
- Linear layer는 저장된 가중치와 편향을 사용하여 입력에 선형 변환을 적용하는 모듈임

In [ ]:
layer1 =nn.Linear(in_features=28*28, out_features=20) # 첫 번째 선형 계층 정의
hidden1 = layer1(flat_image) # 선형 계층에 평탄화된 이미지 전달
print(hidden1.size())

torch.Size([3, 20])


__nn.ReLU__
- 비선형 활성화는 모델의 입출력 사이에 복잡한 관계를 만듬(mapping)
- 선형 변환 후에 적용되어 비선형을 도입 -> 신경망이 다양한 현상을 학습할 수 있게 도움

In [ ]:
print(f"Before: {hidden1}\n\n")
hidden1_relu = nn.ReLU()(hidden1) # ReLU 활성화 함수 적용
print(f"After ReLU: {hidden1_relu}")

Before: tensor([[ 0.0279,  0.2267, -0.1403,  0.4419,  0.1716,  0.0406, -0.1774,  0.0820,
         -0.1654, -0.1746,  0.5445, -0.1549,  0.1933, -0.2561,  0.1548,  0.0231,
          0.6153, -0.0328,  0.5105,  0.1321],
        [-0.0647,  0.4850, -0.0555,  0.4798,  0.3023, -0.0213,  0.1845, -0.1957,
         -0.2536, -0.2656,  0.5379,  0.0474,  0.0226, -0.2780, -0.1177, -0.0620,
          0.3027,  0.0742,  0.4065,  0.1251],
        [-0.2840,  0.3116, -0.0407,  0.3186,  0.0159, -0.0411, -0.0310, -0.0271,
         -0.4712,  0.1968,  0.5833, -0.2276, -0.0097, -0.2783,  0.1700, -0.4551,
          0.4227,  0.3454,  0.6226,  0.0990]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0279, 0.2267, 0.0000, 0.4419, 0.1716, 0.0406, 0.0000, 0.0820, 0.0000,
         0.0000, 0.5445, 0.0000, 0.1933, 0.0000, 0.1548, 0.0231, 0.6153, 0.0000,
         0.5105, 0.1321],
        [0.0000, 0.4850, 0.0000, 0.4798, 0.3023, 0.0000, 0.1845, 0.0000, 0.0000,
         0.0000, 0.5379, 0.0474, 0.0226, 0.0000, 0.0000, 0

__nn.Sequential__
- 순서를 갖는 모듈의 컨테이너이다
- 데이터는 정의된 것과 같은 순서로 모든 모듈들을 통해 전달

In [ ]:
seq_modules = nn.Sequential( # nn.Sequential 계층 정의
    flatten, 
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3, 28, 28) # 임의의 입력 이미지 생성
logits = seq_modules(input_image) # 순차 모델에 입력 이미지 전달

__nn.Softmax__
- 신경망의 마지막 선형 계층은 `nn.Softmax` 모듈에 전달될 범위의 원시값인 logits을 반환함
- logits는 모델의 각 분류에 대한 예측 확률을 나타내도록 [0, 1] 범위로 비례하여 scale됨

In [ ]:
softmax = nn.Softmax(dim=1) # nn.Softmax 계층 초기화
pred_probab = softmax(logits) # 예측 확률 계산

__Model Parameters__
- 신경망 내부의 많은 계층들은 매개변수화 됨
    - 즉, 학습 중 최적화되는 가중치와 편향과 연관지어 짐
- `nn.Module`을 상속하면 모델 객체 내부의 모든 필드들이 자동으로 추적(track)되며, 모델의 `parameters()` 및 `named_parameteres()` 메소드로 모든 매개변수에 접근할 수 있음

In [62]:
print(f"Model structure: {model}\n\n") # 모델 구조 출력

for name, param in model.named_parameters(): # 모델 매개변수 출력
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n") 

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0166,  0.0319,  0.0179,  ..., -0.0332,  0.0351, -0.0094],
        [ 0.0069, -0.0243, -0.0200,  ...,  0.0092,  0.0085, -0.0172]],
       device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0092,  0.0306], device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[-0.0433, -0.0390,  0.0326,  ..., -0.0299,  0.0260, -0.0209],
        [ 0.0061,  0.0029,  0.0215,  ..., -0.0240, -0.0179,  0.0088]],
       device='mps:0', grad_fn=<Slice

In [ ]:
|